# EV Battery Health Prediction: Step 6, Building the Streamlit App

In Steps 3 to 5 I trained models inside notebooks. A notebook is good for experiments, but nobody can use my models from it. In this step I turn the project into a small **web app** where anyone can enter battery measurements and get a predicted **capacity, SOH and RUL**.

| Step | What I do |
|---|---|
| 1 | Load the NASA data for one battery, calculate SOH, plot the capacity fade |
| 2 | Combine 4 batteries and explore the data |
| 3 | Train models to predict SOH |
| 3b | Compare predicting SOH directly with predicting capacity first |
| 4 | Explain the model with SHAP |
| 5 | Predict Remaining Useful Life (RUL) |
| **6 (this notebook)** | **Save the final models and build the Streamlit app** |
| 7 | Deploy the app, write the README, share the project |

**What changed because of Step 3b:** the app no longer predicts SOH directly. It predicts the battery's **capacity in Ah** and converts that to SOH using a **starting capacity** that the user can enter. In Step 3b this was clearly more accurate than predicting SOH directly.

**The files this notebook creates**

| File | Purpose |
|---|---|
| `models/` folder + `sample_input.csv` | The trained models and some example data, saved from the training cells below |
| `app.py` | The Streamlit app itself |
| `requirements.txt` | The list of Python libraries the app needs, so it can be deployed later |

Streamlit apps have to be plain `.py` files, so the `app.py` and `requirements.txt` cells use the `%%writefile` command. That command writes the cell's text into a file instead of running it.

## 1. Why save the models?

A notebook trains a model and then forgets it as soon as the kernel stops. The app cannot retrain on every visit (that would be slow, and it would need the raw data), so I train **once**, save the models to disk, and let the app simply **load** them.

I save them in XGBoost's own **JSON format** instead of pickle. JSON files keep working when library versions change, which matters when the app is deployed on a server with newer packages.

## 2. Importing libraries

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

## 3. Settings

- End of life (EOL) is the first cycle where SOH drops to **80%** or below, the same rule as in Step 5.
- The rated capacity of these cells is about **2.0 Ah**. The app uses it as the default starting capacity when the user does not know the real one.
- The model inputs are the six discharge features. I leave out `cycle`, because a real EV does not always know its cycle count.

In [2]:
EOL_SOH = 80          # end-of-life threshold in SOH %, same as Step 5
RATED_CAPACITY = 2.0  # rated capacity of the cells in Ah

FEATURES = [
    "avg_voltage",
    "min_voltage",
    "avg_current",
    "avg_temp",
    "max_temp",
    "discharge_time_s",
]

Path("models").mkdir(exist_ok=True)

## 4. Loading the data and creating the RUL labels

The RUL labels are created the same way as in Step 5: for each battery, find the first cycle where SOH is at or below the threshold, keep the cycles up to that point, and count down.

In [3]:
df = pd.read_csv("all_batteries.csv").sort_values(["battery", "cycle"]).reset_index(drop=True)

frames = []
for name, g in df.groupby("battery"):
    below = g[g["soh_percent"] <= EOL_SOH]
    if below.empty:
        print(f"{name}: never reached {EOL_SOH}% SOH, left out of the RUL model")
        continue
    eol_cycle = below["cycle"].iloc[0]
    g = g[g["cycle"] <= eol_cycle].copy()
    g["rul"] = eol_cycle - g["cycle"]
    frames.append(g)

rul_df = pd.concat(frames, ignore_index=True)

print("Capacity model rows:", len(df))
print("RUL model rows:", len(rul_df))

Capacity model rows: 636
RUL model rows: 361


## 5. Training the final models

In Steps 3, 3b and 5 I **held one battery out** to measure honest performance. Those held-out scores are the ones I report in my results.

For the app I train on **all four batteries**, because there is very little data and the final model should learn from as much of it as possible. I cannot measure the app model's accuracy on new data, since it has seen every battery. That is why the README should quote the held-out scores from the earlier steps, not a score from these models.

| Model | Target | Why |
|---|---|---|
| Capacity model | `capacity_ah` | Step 3b showed that predicting capacity and converting to SOH was more accurate than predicting SOH directly |
| RUL model | `rul` | Same as Step 5 (on raw features, because the app takes one cycle at a time) |

Both are XGBoost with the same settings as before.

In [4]:
def make_model():
    return XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)


capacity_model = make_model().fit(df[FEATURES], df["capacity_ah"])
rul_model = make_model().fit(rul_df[FEATURES], rul_df["rul"])

print("Both models trained")

Both models trained


## 6. Saving the models

In [5]:
capacity_model.save_model("models/capacity_model.json")
rul_model.save_model("models/rul_model.json")

print("Saved models/capacity_model.json and models/rul_model.json")

Saved models/capacity_model.json and models/rul_model.json


## 7. Saving the information the app needs

Besides the models, the app needs a few extra facts:

| Saved item | Why the app needs it |
|---|---|
| Feature names | So the inputs are always in the same order the model was trained on |
| Min, max and median of each feature | To warn the user when they enter a value outside the range the model has seen |
| Rated capacity | The default starting capacity |
| Three example cycles (early, mid, late life), with the battery's starting capacity | So visitors can try the app with one click |
| `sample_input.csv` | A ready-made file for the "upload many cycles" tab |

In [6]:
ranges = {
    f: {
        "min": float(df[f].min()),
        "max": float(df[f].max()),
        "median": float(df[f].median()),
    }
    for f in FEATURES
}

# Example rows come from battery B0018 (the one I used for testing earlier)
example_battery = df[df["battery"] == "B0018"]
if example_battery.empty:
    example_battery = df[df["battery"] == df["battery"].iloc[0]]

example_start = float(example_battery["capacity_ah"].iloc[0])

picks = {
    "Early life": example_battery.iloc[0],
    "Mid life": example_battery.iloc[len(example_battery) // 2],
    "Late life": example_battery.iloc[-1],
}

examples = {
    label: {
        "cycle": int(row["cycle"]),
        "start_capacity": example_start,
        "actual_capacity": float(row["capacity_ah"]),
        "actual_soh": float(row["soh_percent"]),
        "values": {f: float(row[f]) for f in FEATURES},
    }
    for label, row in picks.items()
}

meta = {
    "features": FEATURES,
    "eol_soh": EOL_SOH,
    "rated_capacity": RATED_CAPACITY,
    "example_battery": str(example_battery["battery"].iloc[0]),
    "ranges": ranges,
    "examples": examples,
    "trained_on": sorted(df["battery"].unique().tolist()),
}

with open("models/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

example_battery[["cycle"] + FEATURES].to_csv("sample_input.csv", index=False)

print("Saved models/meta.json and sample_input.csv")
print("Trained on batteries:", meta["trained_on"])
print(f"Example battery starts at {example_start:.3f} Ah")

Saved models/meta.json and sample_input.csv
Trained on batteries: ['B0005', 'B0006', 'B0007', 'B0018']
Example battery starts at 1.855 Ah


## 8. Quick check: do the saved models load and predict?

Before building the app, I reload the models from the saved files, exactly as the app will, and predict for the three example cycles. SOH is the predicted capacity divided by the battery's starting capacity.

These examples come from a battery the models were trained on, so close predictions here only prove that saving and loading worked. They do not prove the model is accurate on new batteries.

In [7]:
check_cap = XGBRegressor()
check_cap.load_model("models/capacity_model.json")
check_rul = XGBRegressor()
check_rul.load_model("models/rul_model.json")

rows = []
for label, ex in examples.items():
    X = pd.DataFrame([ex["values"]])[FEATURES]
    cap = float(np.clip(check_cap.predict(X)[0], 0, None))
    rows.append(
        {
            "Example": label,
            "Cycle": ex["cycle"],
            "Measured capacity (Ah)": round(ex["actual_capacity"], 3),
            "Predicted capacity (Ah)": round(cap, 3),
            "Measured SOH (%)": round(ex["actual_soh"], 1),
            "Predicted SOH (%)": round(cap / ex["start_capacity"] * 100, 1),
            "Predicted RUL (cycles)": round(float(np.clip(check_rul.predict(X)[0], 0, None)), 1),
        }
    )

pd.DataFrame(rows)

,Example,Cycle,Measured capacity (Ah),Predicted capacity (Ah),Measured SOH (%),Predicted SOH (%),Predicted RUL (cycles)
0,Early life,1,1.855,1.854,100.0,99.9,66.3
1,Mid life,67,1.522,1.522,82.1,82.0,9.5
2,Late life,132,1.341,1.347,72.3,72.6,1.5


## 9. The app: how `app.py` is organised

The app is written in **Streamlit**, a Python library that turns a normal script into a web page. There is no HTML or JavaScript involved. Every `st.something(...)` call adds an element to the page.

| Part of `app.py` | What it does |
|---|---|
| `load_models()` | Loads the two saved models and the metadata. It is cached with `@st.cache_resource`, so it only runs once and not on every click. |
| `predict()` | Takes a table of measurements and a starting capacity, and returns capacity, SOH and RUL. Capacity and RUL cannot go below 0, because negative values make no physical sense. SOH is capacity divided by the starting capacity, so it can go above 100% if the starting capacity entered is too low, and the app tells the user when that happens. |
| **Single cycle** tab | A starting-capacity box, number boxes for the six measurements, a dropdown of examples, a warning if a value is outside the training range, and the predicted capacity, SOH, RUL and a health status. |
| **Upload many cycles** tab | Accepts a CSV (or the built-in sample), checks the required columns exist, predicts for every row, draws SOH and RUL curves, and offers the predictions as a download. |
| **About** tab | Explains the data, the models and, importantly, the limitations. |

The next cell writes this code into `app.py`. The first line, `%%writefile app.py`, tells Jupyter to save the cell as a file instead of running it.

In [8]:
%%writefile app.py
import json
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
from xgboost import XGBRegressor

st.set_page_config(page_title="EV Battery Health Predictor", page_icon="🔋", layout="wide")

BASE_DIR = Path(__file__).parent
MODEL_DIR = BASE_DIR / "models"

LABELS = {
    "avg_voltage": "Average voltage (V)",
    "min_voltage": "Minimum voltage (V)",
    "avg_current": "Average current (A)",
    "avg_temp": "Average temperature (°C)",
    "max_temp": "Maximum temperature (°C)",
    "discharge_time_s": "Discharge time (s)",
}


@st.cache_resource
def load_models():
    capacity_model = XGBRegressor()
    capacity_model.load_model(MODEL_DIR / "capacity_model.json")
    rul_model = XGBRegressor()
    rul_model.load_model(MODEL_DIR / "rul_model.json")
    meta = json.loads((MODEL_DIR / "meta.json").read_text(encoding="utf-8"))
    return capacity_model, rul_model, meta


def predict(frame: pd.DataFrame, start_capacity: float):
    """Return capacity (Ah), SOH (% of the starting capacity) and RUL (>= 0 cycles) for each row."""
    X = frame[FEATURES].astype(float)
    capacity = np.clip(capacity_model.predict(X), 0, None)
    soh = capacity / start_capacity * 100
    rul = np.clip(rul_model.predict(X), 0, None)
    return capacity, soh, rul


def health_status(soh: float) -> str:
    if soh >= 90:
        return "Healthy"
    if soh >= meta["eol_soh"]:
        return "Ageing, still above the end-of-life line"
    return "At or past end of life"


try:
    capacity_model, rul_model, meta = load_models()
except Exception as e:
    st.error(
        "Could not load the saved models. Run the Step 6 notebook first "
        "so the `models/` folder exists."
    )
    st.exception(e)
    st.stop()

FEATURES = meta["features"]
EOL = meta["eol_soh"]
RATED = meta["rated_capacity"]
START_HELP = (
    "SOH is measured against this number. Use the capacity the battery had on its first cycle, "
    f"or its rated capacity ({RATED} Ah for these cells) if you do not have it."
)

# ------------------------------------------------------------------ header
st.title("🔋 EV Battery Health Predictor")
st.write(
    "Estimates a lithium-ion battery's **capacity**, **State of Health (SOH)** and "
    "**Remaining Useful Life (RUL)** from measurements taken during one discharge cycle."
)
st.caption(
    f"Models trained on NASA battery aging data ({', '.join(meta['trained_on'])}). "
    f"SOH = predicted capacity ÷ starting capacity. End of life is defined as SOH ≤ {EOL}%."
)

tab_manual, tab_upload, tab_about = st.tabs(["Single cycle", "Upload many cycles", "About"])

# ------------------------------------------------------------------ tab 1
with tab_manual:
    st.subheader("Enter the measurements from one discharge cycle")

    preset = st.selectbox("Start from an example", list(meta["examples"].keys()))
    example = meta["examples"][preset]
    st.caption(
        f"Example from battery {meta['example_battery']}, cycle {example['cycle']} "
        f"(measured capacity {example['actual_capacity']:.3f} Ah, which is "
        f"{example['actual_soh']:.1f}% of its first-cycle capacity). Change any value below."
    )

    start_capacity = st.number_input(
        "Starting capacity of this battery (Ah)",
        min_value=0.5,
        max_value=5.0,
        value=float(example["start_capacity"]),
        step=0.01,
        format="%.3f",
        key=f"start_{preset}",
        help=START_HELP,
    )

    cols = st.columns(3)
    values = {}
    for i, feat in enumerate(FEATURES):
        with cols[i % 3]:
            values[feat] = st.number_input(
                LABELS.get(feat, feat),
                value=float(example["values"][feat]),
                format="%.4f",
                key=f"{feat}_{preset}",
            )

    out_of_range = [
        LABELS.get(f, f)
        for f in FEATURES
        if not (meta["ranges"][f]["min"] <= values[f] <= meta["ranges"][f]["max"])
    ]
    if out_of_range:
        st.warning(
            "These values are outside the range the model was trained on, so the prediction "
            "may be unreliable: " + ", ".join(out_of_range)
        )

    cap_arr, soh_arr, rul_arr = predict(pd.DataFrame([values]), start_capacity)
    cap, soh, rul = float(cap_arr[0]), float(soh_arr[0]), float(rul_arr[0])

    st.divider()
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Predicted capacity", f"{cap:.3f} Ah")
    c2.metric("Predicted SOH", f"{soh:.1f} %")
    c3.metric("Predicted remaining life", f"{rul:.0f} cycles")
    c4.metric("Status", health_status(soh))
    st.progress(int(round(min(max(soh, 0), 100))))

    if soh > 100:
        st.info(
            "SOH is above 100% because the predicted capacity is higher than the starting "
            "capacity you entered. Check that starting capacity."
        )

# ------------------------------------------------------------------ tab 2
with tab_upload:
    st.subheader("Predict for many cycles at once")
    st.write(
        "Upload a CSV with one row per discharge cycle. Required columns: "
        + ", ".join(f"`{f}`" for f in FEATURES)
        + ". An optional `cycle` column is used for the chart axis."
    )

    uploaded = st.file_uploader("CSV file", type="csv")
    use_sample = st.checkbox(f"Use the built-in sample (battery {meta['example_battery']})")
    using_sample = use_sample and uploaded is None

    start_upload = st.number_input(
        "Starting capacity of this battery (Ah)",
        min_value=0.5,
        max_value=5.0,
        value=float(meta["examples"]["Early life"]["start_capacity"]) if using_sample else float(RATED),
        step=0.01,
        format="%.3f",
        key=f"start_upload_{using_sample}",
        help=START_HELP,
    )

    data = None
    if uploaded is not None:
        data = pd.read_csv(uploaded)
    elif use_sample:
        sample_path = BASE_DIR / "sample_input.csv"
        if sample_path.exists():
            data = pd.read_csv(sample_path)
        else:
            st.error("sample_input.csv was not found next to app.py.")

    if data is not None:
        missing = [f for f in FEATURES if f not in data.columns]
        if missing:
            st.error("Missing required columns: " + ", ".join(missing))
        else:
            data = data.dropna(subset=FEATURES).reset_index(drop=True)
            if data.empty:
                st.error("No complete rows were found in the file.")
            else:
                cap_arr, soh_arr, rul_arr = predict(data, start_upload)

                result = data.copy()
                result["Predicted capacity (Ah)"] = cap_arr.round(4)
                result["Predicted SOH (%)"] = soh_arr.round(2)
                result["Predicted RUL (cycles)"] = rul_arr.round(1)
                x_col = "cycle" if "cycle" in result.columns else None
                plot_df = result.set_index(x_col) if x_col else result

                m1, m2, m3, m4 = st.columns(4)
                m1.metric("Cycles", len(result))
                m2.metric("Capacity at last cycle", f"{cap_arr[-1]:.3f} Ah")
                m3.metric("SOH at last cycle", f"{soh_arr[-1]:.1f} %")
                m4.metric("RUL at last cycle", f"{rul_arr[-1]:.0f} cycles")

                left, right = st.columns(2)
                with left:
                    st.markdown("**Predicted SOH (%)**")
                    st.line_chart(plot_df["Predicted SOH (%)"])
                with right:
                    st.markdown("**Predicted RUL (cycles)**")
                    st.line_chart(plot_df["Predicted RUL (cycles)"])

                st.dataframe(result, width="stretch")
                st.download_button(
                    "Download predictions as CSV",
                    result.to_csv(index=False).encode("utf-8"),
                    file_name="battery_predictions.csv",
                    mime="text/csv",
                )

# ------------------------------------------------------------------ tab 3
with tab_about:
    st.subheader("How this works")
    st.markdown(
        f"""
**Data.** NASA Prognostics Center of Excellence Li-ion battery aging dataset: batteries
charged and discharged in a lab until they wore out.

**Inputs.** Six numbers summarising one discharge cycle (voltage, current, temperature, discharge time),
plus the battery's starting capacity.

**Models.** Two XGBoost regressors. One predicts the battery's **capacity in Ah**, and SOH is that
capacity divided by the starting capacity you enter. The other predicts the number of discharge cycles
left until SOH reaches {EOL}%.

**Why capacity and not SOH directly?** The batteries in the dataset start at slightly different
capacities. Predicting SOH directly forces every battery to start at 100%, which blurs that difference
and made the model under-predict early life. Predicting capacity first worked better in testing.

**Limits you should know about**
- Only four batteries were used, all from one lab, so this is a **proof of concept**,
  not a tool for real vehicles.
- SOH depends on the starting capacity you enter. Using the rated capacity instead of the real
  first-cycle capacity shifts every SOH value by the difference between them.
- Discharge time is almost the same information as capacity, so the model works best when a
  full discharge was measured. A real vehicle rarely gives you one.
- The remaining-life prediction is much less accurate than the capacity prediction, especially
  early in a battery's life.
- Predictions outside the training ranges are unreliable (the app warns you).
- The {EOL}% end-of-life line is a common convention, not a universal rule.
"""
    )


Overwriting app.py


## 10. `requirements.txt`

When the app is deployed, the server starts empty, so it needs to know which libraries to install. This file lists them, one per line.

In [9]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
xgboost


Overwriting requirements.txt


## 11. Running the app

A Streamlit app keeps running as a web server, so it cannot be started from inside a notebook cell. I run it from the terminal, in the project folder:

```
python -m pip install streamlit
python -m streamlit run app.py
```

It opens in the browser at `http://localhost:8501`. To stop it, press `Ctrl+C` in the terminal.

**Things I want to test when it opens**
- Switch between the three examples (early, mid and late life) and check that capacity, SOH and RUL all go down.
- Change the starting capacity from the example's value to 2.0 and check that the SOH drops by the right amount (the same capacity is now measured against a bigger starting number).
- Type a value far outside the normal range and check that the warning appears.
- On the upload tab, tick the built-in sample and check that the two curves appear.

## 12. Limitations

- **Very little data.** The models learned from four batteries tested in one lab, so the app is a **proof of concept** and not something to use on a real vehicle.
- **No honest accuracy for these exact models.** They were trained on every battery, so the accuracy numbers I quote come from the held-out tests in Steps 3, 3b and 5.
- **SOH depends on the starting capacity entered.** If the rated capacity (2.0 Ah) is used instead of the battery's real first-cycle capacity, every SOH value shifts by the difference between the two.
- **Discharge time is almost the same information as capacity.** The capacity model works best when a full discharge was measured, which a real vehicle rarely provides.
- **The RUL model is much weaker than the capacity model.** In Step 5 its typical error on the held-out battery was around 11 to 16 cycles, and it was worst early in life. It also looks at one cycle at a time and does not see the battery's history.
- **End of life is a convention.** The 80% line is common for EVs, but it is not a universal rule.

## 13. Next step

In Step 7 I will put the project on GitHub, deploy the app on Streamlit Community Cloud to get a live link, and write the project README.